## Demonstrate the use of `nested_cv`

### Binary problem

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from nested_cv import evaluate_nested_cv

# Load or create your dataset
X, y = load_iris(return_X_y=True)
y = y[50:150]
X = X[50:150]

# Define your estimator
estimator = RandomForestClassifier(random_state=42)

# Define hyperparameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5],
}

# Define metrics
metrics = ['accuracy', 'precision', 'recall', 'f1']

# Call the function
results = evaluate_nested_cv(
    X=X,
    y=y,
    estimator=estimator,
    param_grid=param_grid,
    metrics=metrics,
    refit_metric='f1',  # Which metric to optimize for
    outer_splits=5,
    inner_splits=3,
    is_classification=True,
    random_state=42,
    verbose=True,
    return_fold_estimators=True
)

# Access results
print(results['metrics'])  # Outer fold evaluation scores
print(results['hyperparameter_stability'])  # How consistent params were
print(results['combo_comparison'])  # Comparison of hyperparameter combinations


Starting Nested CV | Optimising for: f1


Fold 01 | accuracy: 0.9000 | precision: 1.0000 | recall: 0.8000 | f1: 0.8889
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1: 0.9497
--------------------------------------------------------------------------------


Fold 02 | accuracy: 1.0000 | precision: 1.0000 | recall: 1.0000 | f1: 1.0000
         Best params      : {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
         Best inner f1: 0.9240
--------------------------------------------------------------------------------


Fold 03 | accuracy: 0.8500 | precision: 0.8182 | recall: 0.9000 | f1: 0.8571
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1: 0.9487
--------------------------------------------------------------------------------


Fold 04 | accuracy: 0.9000 | precision: 0.8333 | recall: 1.0000 | f1: 0.9091
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
         Best inner f1: 0.9505
--------------------------------------------------------------------------------


Fold 05 | accuracy: 1.0000 | precision: 1.0000 | recall: 1.0000 | f1: 1.0000
         Best params      : {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
         Best inner f1: 0.8897
--------------------------------------------------------------------------------
NESTED CV COMPLETE — OUTER GENERALISATION SCORES
  accuracy                  | mean: 0.9300 ± 0.0671 [min: 0.8500, max: 1.0000]
  precision                 | mean: 0.9303 ± 0.0956 [min: 0.8182, max: 1.0000]
  recall                    | mean: 0.9400 ± 0.0894 [min: 0.8000, max: 1.0000]
  f1                        | mean: 0.9310 ± 0.0656 [min: 0.8571, max: 1.0000]
  Most selected params (2/5 folds): {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
{'accuracy': {'fold_scores': [0.9, 1.0, 0.85, 0.9, 1.0], 'mean': 0.93, 'std': 0.0671, 'min': 0.85, 'max': 1.0}, 'precision': {'fold_scores': [1.0, 1.0, 0.8182, 0.8333, 1.0], 'mean': 0.9303, 'std': 0.0956, 'min': 0.8182, 'max': 1.0}, 'recall': {'fold_scores': 

In [2]:
# Outer fold generalization scores
print(results['metrics']['accuracy'])
# Output: {'fold_scores': [...], 'mean': 0.95, 'std': 0.02, ...}

# Best hyperparameters per fold
print(results['best_params_per_fold'])

# Parameter stability analysis
print(results['hyperparameter_stability']['most_frequent'])

# Detailed per-fold information
for fold in results['fold_details']:
    print(f"Fold {fold['fold']}: {fold['best_params']}")

# Trained models (if requested)
if 'fold_estimators' in results:
    best_model = results['fold_estimators'][0]

{'fold_scores': [0.9, 1.0, 0.85, 0.9, 1.0], 'mean': 0.93, 'std': 0.0671, 'min': 0.85, 'max': 1.0}
[{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}, {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}, {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}]
{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 1: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 2: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}
Fold 3: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 4: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
Fold 5: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 100}


In [3]:
results.keys()

dict_keys(['metrics', 'combo_comparison', 'best_params_per_fold', 'hyperparameter_stability', 'fold_details', 'config', 'fold_estimators'])

In [4]:
best_model

RandomForestClassifier(max_depth=5, n_estimators=50, random_state=42)

### Multiclass problem

In [5]:
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score


# Multiclass dataset (iris has 3 classes)
X, y = load_iris(return_X_y=True)

estimator = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5],
}

# Define both macro and weighted averages for comparison
metrics = {
    'accuracy': 'accuracy',
    'f1_macro': make_scorer(f1_score, average='macro', zero_division=0),
    'f1_weighted': make_scorer(f1_score, average='weighted', zero_division=0),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
}

results = evaluate_nested_cv(
    X=X,
    y=y,
    estimator=estimator,
    param_grid=param_grid,
    metrics=metrics,
    refit_metric='f1_macro',  # Use macro F1 for balanced evaluation
    outer_splits=5,
    inner_splits=3,
    is_classification=True,
    random_state=42,
    verbose=True,
    return_fold_estimators=True
)

# Results will now show all multiclass metrics
print(results['metrics']['f1_macro'])
print(results['metrics']['f1_weighted'])


Starting Nested CV | Optimising for: f1_macro


Fold 01 | accuracy: 0.9667 | f1_macro: 0.9666 | f1_weighted: 0.9666 | precision_macro: 0.9697 | recall_macro: 0.9667
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
         Best inner f1_macro: 0.9247
--------------------------------------------------------------------------------


Fold 02 | accuracy: 0.9667 | f1_macro: 0.9666 | f1_weighted: 0.9666 | precision_macro: 0.9697 | recall_macro: 0.9667
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1_macro: 0.9663
--------------------------------------------------------------------------------


Fold 03 | accuracy: 0.9333 | f1_macro: 0.9327 | f1_weighted: 0.9327 | precision_macro: 0.9444 | recall_macro: 0.9333
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1_macro: 0.9493
--------------------------------------------------------------------------------


Fold 04 | accuracy: 1.0000 | f1_macro: 1.0000 | f1_weighted: 1.0000 | precision_macro: 1.0000 | recall_macro: 1.0000
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1_macro: 0.9420
--------------------------------------------------------------------------------


Fold 05 | accuracy: 0.9000 | f1_macro: 0.8997 | f1_weighted: 0.8997 | precision_macro: 0.9024 | recall_macro: 0.9000
         Best params      : {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
         Best inner f1_macro: 0.9667
--------------------------------------------------------------------------------
NESTED CV COMPLETE — OUTER GENERALISATION SCORES
  accuracy                  | mean: 0.9533 ± 0.0380 [min: 0.9000, max: 1.0000]
  f1_macro                  | mean: 0.9531 ± 0.0382 [min: 0.8997, max: 1.0000]
  f1_weighted               | mean: 0.9531 ± 0.0382 [min: 0.8997, max: 1.0000]
  precision_macro           | mean: 0.9572 ± 0.0364 [min: 0.9024, max: 1.0000]
  recall_macro              | mean: 0.9533 ± 0.0380 [min: 0.9000, max: 1.0000]
  Most selected params (4/5 folds): {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
{'fold_scores': [0.9666, 0.9666, 0.9327, 1.0, 0.8997], 'mean': 0.9531, 'std': 0.0382, 'min': 0.8997, 'max': 1.0}
{'fold_scores': [0.966

In [6]:
# Outer fold generalization scores
print(results['metrics']['accuracy'])
# Output: {'fold_scores': [...], 'mean': 0.95, 'std': 0.02, ...}

# Best hyperparameters per fold
print(results['best_params_per_fold'])

# Parameter stability analysis
print(results['hyperparameter_stability']['most_frequent'])

# Detailed per-fold information
for fold in results['fold_details']:
    print(f"Fold {fold['fold']}: {fold['best_params']}")

# Trained models (if requested)
if 'fold_estimators' in results:
    best_model = results['fold_estimators'][0]

{'fold_scores': [0.9667, 0.9667, 0.9333, 1.0, 0.9], 'mean': 0.9533, 'std': 0.038, 'min': 0.9, 'max': 1.0}
[{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}, {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}]
{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 1: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}
Fold 2: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 3: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 4: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Fold 5: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}


In [7]:
best_model

RandomForestClassifier(max_depth=5, random_state=42)